# Lab 1 — Loading and Inspecting Data

**DS331 · Introduction to Data Science**

Welcome to the first lab notebook! Every data science project starts the same way: you get a dataset, you load it into Python, and you *look at it carefully* before doing anything else.

In this notebook you will learn how to:

1. [Import the libraries we need](#1)
2. [Load a dataset (from a URL, a library, or your own file)](#2)
3. [Take a first look at the data](#3)
4. [Select rows and columns](#4)
5. [Count and summarize values](#5)
6. [Preview: missing values](#6)
7. [Try it yourself ✏️](#7)

**Dataset:** the *Palmer Penguins* dataset — body measurements of 344 penguins from three species living on three islands in Antarctica. It is small, easy to understand, and perfect for learning.

<a id="1"></a>
## 1. Importing the libraries

We always start by importing the libraries we will use. The two most important ones for data work are:

- **pandas** — for loading and manipulating tabular data (think "Excel inside Python"). Imported as `pd` by convention.
- **seaborn** — a plotting library that also ships with several practice datasets. Imported as `sns` by convention.

These short aliases (`pd`, `sns`) are universal conventions — every tutorial and Stack Overflow answer uses them, so you should too.

In [ ]:
import pandas as pd
import seaborn as sns

print("pandas version:", pd.__version__)
print("seaborn version:", sns.__version__)

<a id="2"></a>
## 2. Loading a dataset

The object that holds a table in pandas is called a **DataFrame**. There are several ways to create one — here are the three you will use most often.

### 2.1 From a CSV file on the internet

`pd.read_csv()` is *the* workhorse function. You can give it a local file path **or a URL** — it works the same way.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(url)

type(penguins)

### 2.2 From a library

Seaborn bundles the same dataset, so this one line is equivalent — handy for practice:

```python
penguins = sns.load_dataset("penguins")
```

### 2.3 From your own file (what you'll do for your report)

When you work on **your assigned dataset** in Google Colab, you have two options:

```python
# Option A: upload the file directly into the Colab session
from google.colab import files
uploaded = files.upload()                 # a file-picker appears
df = pd.read_csv("your_dataset.csv")

# Option B: mount your Google Drive (file survives between sessions)
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/your_dataset.csv")
```

> ⚠️ **Common mistake:** `pd.read_csv("dataset.csv")` fails with `FileNotFoundError` if the file is not in the current folder. In Colab, uploaded files land in `/content/` — check the 📁 file panel on the left to see what is actually there.

Useful `read_csv` options you may need for your own dataset:

| Argument | When you need it |
|---|---|
| `sep=";"` | columns are separated by `;` instead of `,` |
| `header=None` | the file has no column names in the first row |
| `names=[...]` | provide your own column names |
| `encoding="latin-1"` | you get a `UnicodeDecodeError` |
| `na_values=["?", "N/A"]` | missing values are written as something unusual |

There are also `pd.read_excel()`, `pd.read_json()` and others for non-CSV formats.

<a id="3"></a>
## 3. First look at the data

**Never skip this step.** Before any analysis, you must know: How big is the table? What are the columns? What types do they have? Do the values look sensible?

### 3.1 `head()`, `tail()`, `sample()` — peek at the rows

In [ ]:
penguins.head()        # first 5 rows (pass a number for more, e.g. head(10))

In [ ]:
penguins.tail(3)       # last 3 rows

In [ ]:
penguins.sample(5, random_state=42)   # 5 random rows — good for spotting surprises

Each row is one penguin. The columns mix **numeric** measurements (bill length, flipper length, body mass) and **categorical** labels (species, island, sex). Most real datasets look like this.

### 3.2 `shape`, `columns`, `dtypes` — the table's structure

Note: these are **attributes**, not methods — no parentheses `()`.

In [ ]:
print("shape   :", penguins.shape)        # (rows, columns)
print("rows    :", penguins.shape[0])
print("columns :", list(penguins.columns))

In [ ]:
penguins.dtypes

- `object` usually means **text** (strings) — here the categorical columns.
- `float64` / `int64` are **numbers**.

> ⚠️ **Common mistake:** if a column that *should* be numeric (e.g. a price) shows up as `object`, something in it is not a number — maybe a `$` sign, a comma, or a `"?"` used for missing values. We will fix exactly this kind of problem in Lab 2.

### 3.3 `info()` — structure + missing values in one view

In [ ]:
penguins.info()

Read this output carefully — it answers three questions at once:

1. **344 entries** → number of rows.
2. **Non-Null Count** → e.g. `bill_length_mm` has 342 non-null values out of 344, so **2 values are missing**. The `sex` column is missing 11.
3. **Dtype** → the type of each column.

### 3.4 `describe()` — summary statistics

In [ ]:
penguins.describe()

For every **numeric** column you get the count, mean, standard deviation, min/max, and the quartiles (25%, 50% = median, 75%).

Sanity-check these numbers against common sense: body mass between 2700 g and 6300 g is plausible for penguins. If you saw a minimum of `-999`, that would scream "missing-value code"!

By default `describe()` ignores text columns. Ask for them by excluding the numeric ones:

In [ ]:
penguins.describe(exclude="number")   # count, unique values, most frequent value

<a id="4"></a>
## 4. Selecting rows and columns

You constantly need to zoom in on a part of the table. There are a few core patterns — learn these well.

### 4.1 Selecting columns

In [ ]:
penguins["species"].head()            # one column -> a Series (a single column object)

In [ ]:
penguins[["species", "body_mass_g"]].head()   # note the DOUBLE brackets: a list of columns -> DataFrame

> ⚠️ **Common mistake:** `penguins["species", "body_mass_g"]` (single brackets, two names) is a `KeyError`. To select multiple columns you pass a **list**, hence the double brackets.

### 4.2 Selecting rows by position — `iloc`

`iloc` = **i**nteger **loc**ation. It works like list indexing: `df.iloc[row_positions, column_positions]`.

In [ ]:
penguins.iloc[0]          # the first row

In [ ]:
penguins.iloc[10:15, 0:3]   # rows 10–14, first three columns

### 4.3 Selecting rows by label/condition — `loc`

`loc` uses **labels** (index values and column names): `df.loc[rows, columns]`.

In [ ]:
penguins.loc[0:4, ["species", "island", "sex"]]

### 4.4 Boolean filtering — the most useful pattern in pandas

Write a condition on a column → get a True/False mask → use it to keep only matching rows.

In [ ]:
heavy = penguins[penguins["body_mass_g"] > 5000]
print("Penguins heavier than 5 kg:", len(heavy))
heavy.head()

In [ ]:
# Combining conditions: & = and, | = or, ~ = not.
# EACH condition needs its own parentheses!
heavy_gentoo_females = penguins[
    (penguins["species"] == "Gentoo")
    & (penguins["sex"] == "Female")
    & (penguins["body_mass_g"] > 5000)
]
heavy_gentoo_females.head()

> ⚠️ **Common mistakes** with filtering:
> - Using Python's `and` / `or` instead of `&` / `|` → `ValueError: The truth value of a Series is ambiguous`.
> - Forgetting the parentheses around each condition → confusing operator-precedence errors.
> - Using `=` (assignment) instead of `==` (comparison).

### 4.5 `query()` — the same thing, written as text

Some people find this more readable; both styles are fine.

In [ ]:
penguins.query("species == 'Gentoo' and body_mass_g > 5000").head()

<a id="5"></a>
## 5. Counting and summarizing

### 5.1 `value_counts()` — how often does each value appear?

This is *the* first tool for exploring a categorical column.

In [ ]:
penguins["species"].value_counts()

In [ ]:
penguins["island"].value_counts(normalize=True).round(3)   # as proportions instead of counts

In [ ]:
print("unique values:", penguins["species"].unique())
print("how many     :", penguins["species"].nunique())

### 5.2 Statistics of a single column

In [ ]:
col = penguins["body_mass_g"]
print("mean   :", round(col.mean(), 1))
print("median :", col.median())
print("std    :", round(col.std(), 1))
print("min/max:", col.min(), "/", col.max())

### 5.3 `sort_values()` — order the table

In [ ]:
penguins.sort_values("body_mass_g", ascending=False).head(5)   # the 5 heaviest penguins

### 5.4 `groupby()` — statistics *per group*

This is one of the most powerful ideas in pandas: **split** the data into groups, **apply** a calculation to each group, and **combine** the results.

Question: *what is the average body mass of each species?*

In [ ]:
penguins.groupby("species")["body_mass_g"].mean().round(1)

In [ ]:
# Several statistics at once with .agg()
penguins.groupby("species")["body_mass_g"].agg(["count", "mean", "min", "max"]).round(1)

In [ ]:
# Grouping by two columns
penguins.groupby(["species", "sex"])["body_mass_g"].mean().round(1)

Already a finding for a report: *Gentoo penguins are clearly heavier than the other two species, and within every species males are heavier than females.* This is what EDA is about — turning simple summaries into statements about the data.

<a id="6"></a>
## 6. Preview: missing values

`info()` hinted that some values are missing. Here is the standard one-liner to count them per column — we will deal with them properly in **Lab 2**.

In [ ]:
penguins.isna().sum()

<a id="7"></a>
## 7. Try it yourself ✏️

Practice on a different dataset: **tips** — restaurant bills and tips. Load it and answer the questions below. (Solutions are not provided on purpose — everything you need is in this notebook.)

In [ ]:
tips = sns.load_dataset("tips")
tips.head()

**Exercise 1.** How many rows and columns does `tips` have? Which columns are numeric and which are categorical? Are any values missing?

**Exercise 2.** Show summary statistics for the numeric columns. What is the largest total bill?

**Exercise 3.** How many smokers vs. non-smokers are in the data? (Hint: `value_counts`.)

**Exercise 4.** Select all rows where the bill is above 40 dollars *and* the party size is at least 4. How many are there?

**Exercise 5.** Using `groupby`, compute the average tip for each day of the week. Which day tips best?

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

---
**Next lab:** the data is rarely as tidy as the penguins. In **Lab 2 — Data Cleaning and Preprocessing** we work with the Titanic dataset and deal with missing values, wrong types, duplicates and outliers.